# Linear observables of the unitary QELM: distributions per charge sector

Issue #191. The standard unitary QELM, with no measurement during the evolution (`p = 0`): the experiment `experiments/QELM_Q/exp_QELM_Q.py` at `projective_measurement.p = 0` and one evolution step (`nbr_time_evolution = 1`), which is exactly one unitary `exp(-i H t)` per sample (the `p = 0` baseline checked in notebook 04). For each sample:

1. The ancilla `R` starts entangled with the charge of the `N` system qubits, `(|0>_R |Q1>_S + |1>_R |Q2>_S) / sqrt(2)`, with `Q1 = N // 2` and `Q2 = Q1 - 1`. `R` is measured in Z first (Born rule, probability 1/2 each): the outcome puts the system in the **charge sector** `Q1` or `Q2`, uniform superposition of the configurations of that charge, and the label of the sample is that charge, `Y = Q`.
2. A new disordered TFIM `H = h sum_i Z_i + sum_{i<j} J_ij X_i X_j` on the system qubits (`R` does not evolve) acts for the time `t`.
3. The features are the **linear observables** `<O_k> = Tr(O_k rho)` of the system qubits, for `O = Z, Y, X`, plus a bias.

Over a dataset, each `<O_k>` is a random variable in `[-1, 1]`. This notebook:

1. generates independent datasets of increasing size `n_samples` (100, 500, 2000, 5000 by default), for a strong (`h = 10 J`) and a weak (`h = J`) field;
2. isolates the linear observables (`QELM.evaluation.feature_distribution.linear_observables`);
3. fits the ridge readout of `Q` on them, all three together and each alone, for their baseline predictive power;
4. superimposes the histograms of `<Z_{N/2}>` for the different `n_samples`, separately for the samples of charge `Q1`, of charge `Q2`, and for both together (`QELM.utils.plots.draw_histograms`), with the Gaussian fit of each panel;
5. measures the convergence of the empirical laws with `n_samples` and analyzes their final shape.

**Symmetries.** Every term of `H` conserves the parity `prod_i Z_i` of the system, and each charge sector has a definite parity. A single `X_k` or `Y_k` flips one spin, so it maps the state to the other parity: `<X_k> = <Y_k> = 0` **exactly**, for every sample and every `h`. Only `<Z_k>` carries information; the histograms show it alone.

**Gaussian fits.** On each sector panel, the Gaussian of the mean and standard deviation of the largest dataset of that sector: the class-conditional density that a quadratic discriminant analysis (QDA) of the charge would fit from one feature. On the combined panels, the mixture of the two class Gaussians weighted by the class frequencies, the marginal density of that QDA model.

**Qubit counts (issue #235).** `N` is the number of system qubits, the ancilla excluded; the simulated state has `N_total = N + 1` qubits. The sizes of this notebook, its defaults and the results quoted in its text were moved to this convention: a result obtained at `N = 6` before issue #235, when `N` counted the ancilla, is quoted here at `N = 5`, the same system.

In [ ]:
import os
from pathlib import Path

# Each sample is a small dense evolution: a single thread is fastest
# (notebook 06 measured it), and must be set before NumPy is imported.
os.environ.setdefault("OMP_NUM_THREADS", "1")

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_q_linear_notebook.py` does for a small run). `N` is the number of system qubits, ancilla excluded (issue #235); `H_VALUES` are the transverse fields `TFIM.h` in units of `TFIM.J = 1`; `N_SAMPLES` are the dataset sizes. The dataset of the `k`-th size uses the seed `SEED + k`, so the datasets are independent: the distance between two of them measures the sampling noise only. `SITE` is the system qubit whose `<Z>` is plotted, `N // 2` by default (chain site `SITE + 1`, after the ancilla). Each run goes to this notebook's own folder, `outputs/QELM_Q_linear/<date_time>/`; `RUN = False` reads back the latest one without simulating again.

With the defaults, `N = 5` and `h = 10 J, 1 J`, the 2 x 7600 samples take about 5 minutes.

In [ ]:
N = int(os.environ.get("N", "5"))  # system qubits, ancilla excluded
H_VALUES = [float(h) for h in os.environ.get("H_VALUES", "10 1").split()]
N_SAMPLES = [int(n) for n in os.environ.get("N_SAMPLES", "100 500 2000 5000").split()]
SEED = int(os.environ.get("SEED", "42"))
SITE = int(os.environ.get("SITE", str(N // 2)))  # system qubit, 0 .. N-1
BINS = int(os.environ.get("BINS", "40"))
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"
LABELS = ["Z", "Y", "X"]  # the linear feature sets of exp_QELM_Q

## Generate the datasets

One run of `exp_QELM_Q.run` per field `h` and size `n_samples`, in this process, with `samples.L = n_samples` (70 % train, 30 % test, as in the config), `p = 0` and a single evolution step. Each run is saved in its own subfolder `h<h>_n<n_samples>/` of the sweep folder, with `exp_QELM_Q.npz` (the feature sets `features_<name>`, bias column included, the charges `Y`), `metrics.json` and `resolved_config.json`. The cell then reads the runs back from that folder, whether it just wrote them or not.

In [ ]:
import importlib.util
import json

import numpy as np

from QELM import load_config
from QELM.utils import latest_sweep, new_sweep_dir, save_config, save_results

spec = importlib.util.spec_from_file_location(
    "exp_QELM_Q", Path("experiments/QELM_Q/exp_QELM_Q.py")
)
assert spec is not None and spec.loader is not None
exp_QELM_Q = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exp_QELM_Q)

EXPERIMENT = "QELM_Q_linear"  # this notebook's own output folder (issue #151)


def overrides(h: float, k: int, n: int) -> list[str]:
    return [
        "models=TFIM",
        f"N={N}",
        f"TFIM.h={h}",
        f"seed={SEED + k}",
        f"samples.L={n}",
        # The unitary QELM: no measurement, one step exp(-i H t).
        "projective_measurement.p=0.0",
        "projective_measurement.nbr_time_evolution=1",
    ]


if RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, EXPERIMENT)
    for h in H_VALUES:
        for k, n in enumerate(N_SAMPLES):
            print(f"h = {h:g} J, n_samples = {n}")
            cfg = load_config(overrides(h, k, n))
            out = exp_QELM_Q.run(cfg)
            assert out["n_measured_mean"] == 0
            run_dir = sweep_dir / f"h{h:g}_n{n}"
            save_results(
                out, run_dir, "exp_QELM_Q", extra_metrics=exp_QELM_Q.EXTRA_METRICS
            )
            save_config(cfg, run_dir)
else:
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

# Read back every run of the sweep folder.
runs = []
for run_dir in sorted(p for p in sweep_dir.iterdir() if p.is_dir()):
    cfg = json.loads((run_dir / "resolved_config.json").read_text())
    data = np.load(run_dir / "exp_QELM_Q.npz")
    runs.append(
        {
            "h": float(cfg["TFIM"]["h"]),
            "n_samples": int(cfg["samples"]["L"]),
            "seed": int(cfg["seed"]),
            "N": int(cfg["N"]),
            "train_size": int(cfg["samples"]["train_size"]),
            "features": {label: data[f"features_{label}"] for label in LABELS},
            "Y": data["Y"],
            "metrics": json.loads((run_dir / "metrics.json").read_text()),
            "cfg": cfg,
        }
    )
runs.sort(key=lambda r: (-r["h"], r["n_samples"]))
hs = sorted({r["h"] for r in runs}, reverse=True)
sizes = sorted({r["n_samples"] for r in runs})
Q1, Q2 = N // 2, N // 2 - 1  # the two charge sectors of the system
# Q2 = 0 is a single configuration, a fixed <Z_k> = 1: no law to study.
assert Q2 >= 1, f"N >= 4 is needed for two nontrivial sectors, got N = {N}."
assert all(r["N"] == N for r in runs), "the sweep was run with another N"
assert all(set(np.unique(r["Y"])) <= {Q1, Q2} for r in runs)
print(f"{len(runs)} runs: h = {hs}, n_samples = {sizes}, N = {N}, Q = {Q1}, {Q2}")

## Isolate the linear observables

`linear_observables` turns each feature set into a block of shape `(n_samples, N)` and drops its bias column. The largest `|<X_k>|` and `|<Y_k>|` over all samples and sites check the parity argument above: they must be zero up to rounding. The fraction of samples of charge `Q1` is 1/2 up to the sampling noise.

In [ ]:
from QELM.evaluation.feature_distribution import linear_observables

for r in runs:
    r["obs"] = {
        label: linear_observables(r["features"][label], [label], N)[label]
        for label in LABELS
    }

for h in hs:
    rows_h = [r for r in runs if r["h"] == h]
    largest = {
        label: max(float(np.max(np.abs(r["obs"][label]))) for r in rows_h)
        for label in LABELS
    }
    frac = np.mean(np.concatenate([r["Y"] for r in rows_h]) == Q1)
    print(
        f"h = {h:g} J: max |<Z_k>| = {largest['Z']:.3f}, "
        f"max |<Y_k>| = {largest['Y']:.1e}, max |<X_k>| = {largest['X']:.1e}, "
        f"fraction of Q = {Q1}: {frac:.3f}"
    )

## Baseline predictive power of the linear features

The ridge readout of `exp_QELM_Q` (no intercept, hence the bias column; `training.alpha` of `conf/training/regression/ridge.yaml`) predicts the charge `Q` from the linear features: all three Paulis together, `[<Z_k>, <Y_k>, <X_k>, 1]`, and each alone, with the same train/test split (the first `train_size` samples train). The accuracy counts a prediction right when it falls on the side of `(Q1 + Q2) / 2` of the true charge; 0.5 is chance. A feature set that is identically 0 leaves only the bias, and the readout always predicts the majority charge of the train set.

Saved as `ridge_vs_n_samples.png` and `.json` in the sweep folder.

In [ ]:
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from QELM.evaluation.metrics import R2, accuracy
from QELM.training import RidgeReadout
from QELM.utils.plots import color_map, save_figure, subplots

FEATURE_SETS = {"Z, Y, X": LABELS, "Z": ["Z"], "Y": ["Y"], "X": ["X"]}
THRESHOLD = (Q1 + Q2) / 2


def ridge_scores(r: dict, labels: list[str]) -> tuple[float, float]:
    features = np.hstack([r["obs"][label] for label in labels])
    features = np.hstack([features, np.ones((features.shape[0], 1))])
    train = r["train_size"]
    solver = RidgeReadout.from_config(OmegaConf.create(r["cfg"]))
    solver.fit(features[:train], r["Y"][:train])
    pred = solver.predict(features[train:])
    return accuracy(r["Y"][train:], pred, THRESHOLD), R2(r["Y"][train:], pred)


ridge_table = []
for r in runs:
    row = {"h": r["h"], "n_samples": r["n_samples"]}
    for name, labels in FEATURE_SETS.items():
        row[f"accuracy_{name}"], row[f"r2_{name}"] = ridge_scores(r, labels)
    # Refitting on <Z_k> alone reproduces the run's own score.
    assert np.isclose(row["accuracy_Z"], r["metrics"]["accuracy_Z"])
    ridge_table.append(row)

print(
    f"{'h':>5} {'n':>6} "
    + " ".join(f"{'acc ' + name:>11} {'R2 ' + name:>10}" for name in FEATURE_SETS)
)
for row in ridge_table:
    print(
        f"{row['h']:>5g} {row['n_samples']:>6} "
        + " ".join(
            f"{row[f'accuracy_{name}']:>11.3f} {row[f'r2_{name}']:>10.3f}"
            for name in FEATURE_SETS
        )
    )

fig_r2, axes_r2 = subplots(1, len(hs), figsize=(6.5 * len(hs), 5), squeeze=False)
set_colors = color_map(FEATURE_SETS)
for h, ax in zip(hs, axes_r2[0], strict=True):
    rows_h = [row for row in ridge_table if row["h"] == h]
    for name in FEATURE_SETS:
        ax.plot(
            [row["n_samples"] for row in rows_h],
            [row[f"accuracy_{name}"] for row in rows_h],
            "o-",
            color=set_colors[name],
            label=f"<{name}>" if len(name) == 1 else "<Z>, <Y>, <X>",
        )
    ax.axhline(0.5, color="gray", linestyle=":", label="chance")
    ax.set(
        xscale="log",
        xlabel="n_samples",
        ylabel="test accuracy on Q",
        title=f"ridge on linear observables, h = {h:g} J",
    )
    ax.legend()
fig_r2.suptitle(f"Unitary QELM, N = {N}: baseline of the linear features")
ridge_path = save_figure(
    fig_r2, sweep_dir, "ridge_vs_n_samples", data={"table": ridge_table}
)
print(f"figure: {ridge_path}")
plt.show()

## Superimposed histograms of `<Z_SITE>` per charge sector

One row per field; three columns: the samples of charge `Q1`, those of charge `Q2`, and all of them. In each panel, the density histogram of `<Z>` on system qubit `SITE` for every `n_samples`, with common bins. Each panel shows one Gaussian fit (black, from the largest dataset; on the combined panel, the mixture of the two class Gaussians). `<Y>` and `<X>` are not drawn: they are exactly 0 (checked above).

Saved as `histograms_linear_observables.png` and `.json`.

In [ ]:
from QELM.utils.plots import common_edges, draw_histograms

PANELS = [f"Q = {Q1}", f"Q = {Q2}", "both"]


def sector_values(r: dict, panel: str) -> np.ndarray:
    values = r["obs"]["Z"][:, SITE]
    if panel == "both":
        return values
    return values[r["Y"] == int(panel.split("= ")[1])]


def gaussian(z: np.ndarray, values: np.ndarray) -> np.ndarray:
    mu, sigma = float(np.mean(values)), float(np.std(values, ddof=1))
    return np.exp(-((z - mu) ** 2) / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))


size_labels = {n: f"n = {n}" for n in sizes}
size_colors = color_map(size_labels.values())
panels = PANELS
fig_h, axes_h = subplots(
    len(hs), len(panels), figsize=(6.5 * len(panels), 5 * len(hs)), squeeze=False
)
z_grid = np.linspace(-1, 1, 801)
for h, row_axes in zip(hs, axes_h, strict=True):
    rows_h = {r["n_samples"]: r for r in runs if r["h"] == h}
    largest = rows_h[sizes[-1]]
    for panel, ax in zip(panels, row_axes, strict=True):
        samples = {size_labels[n]: sector_values(rows_h[n], panel) for n in sizes}
        edges = common_edges(list(samples.values()), BINS)
        handles = draw_histograms(ax, samples, size_colors, edges=edges, kde=False)
        if panel == "both":
            # The QDA marginal: the class Gaussians weighted by their frequency.
            density = sum(
                np.mean(largest["Y"] == q)
                * gaussian(z_grid, sector_values(largest, f"Q = {q}"))
                for q in (Q1, Q2)
            )
            fit_label = "Gaussian mixture of the two classes (largest n)"
        else:
            density = gaussian(z_grid, sector_values(largest, panel))
            fit_label = "Gaussian fit (largest n)"
        (fit,) = ax.plot(z_grid, density, color="k", lw=1.5, label=fit_label)
        ax.set_xlim(-1.02, 1.02)
        title = "Q1 and Q2" if panel == "both" else panel
        ax.set(
            xlabel=f"<Z_{SITE + 1}>",
            ylabel="density",
            title=f"<Z> on system qubit {SITE}, {title}, h = {h:g} J",
        )
        ax.legend(handles=[*handles, fit], fontsize=9)
fig_h.suptitle(f"Unitary QELM, N = {N}: <Z> per charge sector against n_samples")

## Convergence with `n_samples` and final shape of `<Z_SITE>`

For each field and each charge sector, `convergence_table` gives the moments of every dataset and its two-sample Kolmogorov-Smirnov distance to the largest one. Two independent samples of the same law of sizes `n` and `m` lie at a distance of order `sqrt((n + m) / (n m))` (dashed, here with the sizes of a sector, about half of `n_samples`): the empirical law converges at that rate (Glivenko-Cantelli), and a distance far above it would mean that the datasets do not sample the same law. The other panels show the mean (with its standard error) and the standard deviation against `n_samples`.

The shape table compares the largest dataset, for every system qubit and sector, with the Gaussian (KS distance to the Gaussian of the same mean and std; skewness and excess kurtosis 0). Saved as `convergence_vs_n_samples.png` and `.json`; the histogram figure is saved with the shape table.

In [ ]:
from QELM.evaluation.feature_distribution import (
    convergence_table,
    distribution_moments,
    ks_to_normal,
)

sectors = [f"Q = {Q1}", f"Q = {Q2}"]
convergence = {
    (h, sector): convergence_table(
        {r["n_samples"]: sector_values(r, sector) for r in runs if r["h"] == h}
    )
    for h in hs
    for sector in sectors
}
curve_colors = color_map(f"h = {h:g} J, {s}" for h in hs for s in sectors)
fig_c, axes_c = subplots(1, 3, figsize=(19.5, 5))
for (h, sector), rows_c in convergence.items():
    label = f"h = {h:g} J, {sector}"
    color = curve_colors[label]
    n = np.array([row["n_samples"] for row in rows_c], dtype=float)
    smaller = rows_c[:-1]
    axes_c[0].plot(
        n[:-1],
        [row["ks_to_reference"] for row in smaller],
        "o-",
        color=color,
        label=label,
    )
    axes_c[0].plot(
        n[:-1], [row["ks_bound"] for row in smaller], "--", color=color, lw=1
    )
    axes_c[1].errorbar(
        n,
        [row["mean"] for row in rows_c],
        yerr=[row["sem"] for row in rows_c],
        fmt="o-",
        color=color,
        capsize=3,
        label=label,
    )
    axes_c[2].plot(n, [row["std"] for row in rows_c], "o-", color=color, label=label)
axes_c[0].set(
    xscale="log",
    yscale="log",
    xlabel="n_samples",
    ylabel=f"KS distance to n = {sizes[-1]}",
    title=f"convergence of the law of <Z_{SITE + 1}>",
)
axes_c[1].set(xscale="log", xlabel="n_samples", ylabel="mean", title="mean")
axes_c[2].set(xscale="log", xlabel="n_samples", ylabel="std", title="std")
for ax in axes_c:
    ax.legend(fontsize=8)
fig_c.suptitle(
    f"Unitary QELM, N = {N}: law of large numbers for <Z_{SITE + 1}> per sector"
)

shape_table = []
for h in hs:
    largest = next(r for r in runs if r["h"] == h and r["n_samples"] == sizes[-1])
    for sector in (*sectors, "both"):
        q = None if sector == "both" else int(sector.split("= ")[1])
        mask = np.ones(len(largest["Y"]), bool) if q is None else largest["Y"] == q
        for site in range(N):
            values = largest["obs"]["Z"][mask, site]
            shape_table.append(
                {
                    "h": h,
                    "sector": sector,
                    "site": site,
                    **distribution_moments(values),
                    "ks_to_normal": ks_to_normal(values),
                }
            )

print(f"KS distance of <Z_{SITE + 1}> to the largest dataset (n = {sizes[-1]}):")
for (h, sector), rows_c in convergence.items():
    print(
        f"  h = {h:g} J, {sector}: "
        + ", ".join(
            f"n = {row['n_samples']}: {row['ks_to_reference']:.3f} "
            f"(scale {row['ks_bound']:.3f})"
            for row in rows_c[:-1]
        )
    )
print(f"\nShape of <Z_k> at n = {sizes[-1]}:")
print(f"{'':>31} {'mean':>7} {'std':>6} {'skew':>6} {'ex.kurt':>8} {'KS Gauss':>9}")
for row in shape_table:
    print(
        f"h = {row['h']:>4g} J, {row['sector']:>5}, qubit {row['site']:>2}  "
        f"{row['mean']:>+7.3f} {row['std']:>6.3f} {row['skewness']:>+6.2f} "
        f"{row['excess_kurtosis']:>+8.2f} {row['ks_to_normal']:>9.3f}"
    )

histogram_path = save_figure(
    fig_h,
    sweep_dir,
    "histograms_linear_observables",
    data={"shape": shape_table, "site": SITE},
)
convergence_path = save_figure(
    fig_c,
    sweep_dir,
    "convergence_vs_n_samples",
    data={f"h{h:g}, {sector}": rows for (h, sector), rows in convergence.items()},
)
print(f"\nfigures: {histogram_path}, {convergence_path}")
plt.show()

## Analysis

Numbers of the default run (`N = 5`, `Q1 = 2`, `Q2 = 1`; `SITE = 2`, chain site 3; seeds 42 to 45; sweep `outputs/QELM_Q_linear/2026-10-26_08-16-59/`). They change a little with the seeds; the conclusions below do not.

**`<X_k>` and `<Y_k>` carry nothing.** Over all 2 x 7600 samples, `max |<X_k>| = max |<Y_k>| = 0` exactly, at both fields, as the parity argument requires. A readout on `<Y>` or `<X>` alone sees only the bias, so it always predicts the majority charge of the train set. Its accuracy is 0.49 to 0.51 from `n = 500` on (0.67 at `n = 100`, on 30 test samples), and its R2 is at most 0.

**Baseline of the linear readout.** On `<Z_k>`, the accuracy on `Q` is 1.000 for every `n_samples` and both fields. R2 is 1.000 at `h = 10 J` and 0.97 to 0.98 at `h = J`. At large field the system charge is nearly conserved, so `sum_k <Z_k> = N - 2 Q` and `Q` is an affine function of the features. At `h = J` the couplings move some weight to other charges of the same parity (`Q = 2` to 0 and 4, `Q = 1` to 3 and 5), so this relation is only approximate. The two classes stay separated all the same. One feature alone does not separate them: the two class laws of `<Z_3>` overlap over most of `[-0.3, 1]`. The readout needs all `N` values together.

**Convergence (law of large numbers).** In each sector, the KS distance of `<Z_3>` to the largest dataset follows the sampling scale `sqrt((n + m) / (n m))` computed with the sector sizes. At `n = 2000` it is 0.020 to 0.040 for scales of 0.037 to 0.038, and every distance stays below the 5 % critical value `1.36 x scale`. The independent datasets therefore sample one law per sector, and the empirical laws converge at the `1 / sqrt(n)` rate. At `n = 100` (about 50 samples per sector), the histograms are still ragged and the mean of the sector `Q = 2` is 0.04 off. From `n = 2000` on, the histograms of successive sizes lie on top of each other.

**Converged shape.** Every system qubit has the same law (translation invariance of the disorder on average).

- **Sector `Q = 2`.** At `h = 10 J`: mean 0.19 to 0.21 (`1 - 2 Q / N = 0.2`), std 0.27, skewness -0.09 to -0.35, excess kurtosis about 0. The law is close to its Gaussian fit (KS distance 0.013 to 0.028).
- **Sector `Q = 1`.** At `h = 10 J`: mean 0.59 to 0.61 (`1 - 2 Q / N = 0.6`), std 0.30 to 0.32, skewness about -1.1, excess kurtosis about +1. The law is strongly asymmetric and piles up against the bound `<Z> = 1`. A Gaussian misses this (KS distance about 0.10): it puts weight above 1, where no value can be, and underestimates the peak near 1.
- **Both sectors.** The combined law is the 50 / 50 mixture of the two sector laws: mean 0.40, std 0.35, broad and flat, with excess kurtosis about -0.6. The Gaussian mixture of the two class fits follows it.
- **At `h = J`** the sector laws hardly change. Their means drop slightly (0.18 for `Q = 2`, 0.55 for `Q = 1`), as expected from the leakage to other charges of the same parity, and their shapes stay the same.

So the features are not uniform, and they are close to Gaussian only in the larger sector. Their mean and asymmetry are set by the charge sector that the ancilla measurement selects, `<Z_k>` being centered near `1 - 2 Q / N`.

**For a QDA readout.** The Gaussian fits of each panel are the class-conditional densities that a QDA of `Q` on one feature would use. They are a good model for the sector `Q = 2`, but only a rough one for the sector `Q = 1`, whose law is skewed and piles up against `<Z> = 1`. A QDA on a single `<Z_k>` would therefore misplace part of its decision boundary. On all `N` features the classes are separated anyway, through `sum_k <Z_k>`, which is why the linear readout reaches accuracy 1.